In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

REPEAT_DIR = Path(
    "/home/wangxc1117/STDK_GNA_Research/experiment_report/Weather2K/weather2k_80_10_10/weather2k/rmse_tuning/var4_tkeep1000_k_40_fixedspace0.1_time_train0.8_val0.1_test0.1"
)

SUMMARY_CSV = REPEAT_DIR / "k_40_repeat_runs_summary.csv"
ALL_TRIAL_CSV = REPEAT_DIR / "trial_results" / "all_trial_results.csv"
PRED_DIR = REPEAT_DIR / "saved_predictions"

print("REPEAT_DIR =", REPEAT_DIR)
print("SUMMARY_CSV exists:", SUMMARY_CSV.exists())
print("ALL_TRIAL_CSV exists:", ALL_TRIAL_CSV.exists())
print("PRED_DIR exists:", PRED_DIR.exists())
print("n npz files:", len(list(PRED_DIR.glob("seed_*.npz"))))

if not SUMMARY_CSV.exists():
    raise FileNotFoundError(f"SUMMARY_CSV not found: {SUMMARY_CSV}")

if not PRED_DIR.exists():
    raise FileNotFoundError(f"PRED_DIR not found: {PRED_DIR}")

summary_df = pd.read_csv(SUMMARY_CSV)
npz_files = sorted(PRED_DIR.glob("seed_*.npz"))

print("\nsummary_df.shape =", summary_df.shape)
print("n npz files      =", len(npz_files))

if len(npz_files) == 0:
    raise FileNotFoundError(f"No seed_*.npz found in: {PRED_DIR}")

def mean_se(series):
    s = pd.Series(series).dropna().astype(float)
    return float(s.mean()), float(s.sem())

def fmt_mean_se(series, digits=6, scale=1.0):
    m, se = mean_se(pd.Series(series) * scale)
    return f"{m:.{digits}f} ({se:.{digits}f})"

rows = []

for fp in npz_files:
    d = np.load(fp)

    seed = int(d["seed"][0])
    test_idx = d["test_time_idx_run"]

    y_true = d["y_true_raw_run"]
    y_stdk = d["y_stdk_raw_run"]
    y_unreg = d["y_final_unreg_raw_run"]
    y_reg = d["y_final_reg_best_raw_run"]

    rows.append({
        "seed": seed,
        "stdk_rmse_test_raw_recalc": rmse_pooled(y_true[test_idx], y_stdk[test_idx]),
        "unreg_rmse_test_raw_recalc": rmse_pooled(y_true[test_idx], y_unreg[test_idx]),
        "reg_best_rmse_test_raw_recalc": rmse_pooled(y_true[test_idx], y_reg[test_idx]),
        "stdk_mae_test": mae_pooled(y_true[test_idx], y_stdk[test_idx]),
        "unreg_mae_test": mae_pooled(y_true[test_idx], y_unreg[test_idx]),
        "reg_best_mae_test": mae_pooled(y_true[test_idx], y_reg[test_idx]),
        "stdk_r2_test": r2_pooled(y_true[test_idx], y_stdk[test_idx]),
        "unreg_r2_test": r2_pooled(y_true[test_idx], y_unreg[test_idx]),
        "reg_best_r2_test": r2_pooled(y_true[test_idx], y_reg[test_idx]),
        "stdk_covfrob_test_raw_recalc": cov_frob_observed(y_true[test_idx], y_stdk[test_idx]),
        "unreg_covfrob_test_raw_recalc": cov_frob_observed(y_true[test_idx], y_unreg[test_idx]),
        "reg_best_covfrob_test_raw_recalc": cov_frob_observed(y_true[test_idx], y_reg[test_idx]),
    })

extra_df = pd.DataFrame(rows).sort_values("seed").reset_index(drop=True)

merged_df = pd.merge(
    summary_df,
    extra_df,
    on="seed",
    how="inner"
).sort_values("seed").reset_index(drop=True)

print("\nmerged_df.shape =", merged_df.shape)

for a, b in [
    ("stdk_rmse_test_raw", "stdk_rmse_test_raw_recalc"),
    ("unreg_rmse_test_raw", "unreg_rmse_test_raw_recalc"),
    ("reg_best_rmse_test_raw", "reg_best_rmse_test_raw_recalc"),
]:
    if a in merged_df.columns and b in merged_df.columns:
        diff = np.abs(merged_df[a] - merged_df[b]).max()
        print(f"max |{a} - {b}| = {diff:.12f}")

result_table = pd.DataFrame({
    "Split": ["80/10/10", "80/10/10", "80/10/10"],
    "Model": ["STDK", "Adapter (unreg.)", "Adapter (reg.)"],
    "RMSE": [
        fmt_mean_se(merged_df["stdk_rmse_test_raw"]),
        fmt_mean_se(merged_df["unreg_rmse_test_raw"]),
        fmt_mean_se(merged_df["reg_best_rmse_test_raw"]),
    ],
    "MAE": [
        fmt_mean_se(merged_df["stdk_mae_test"]),
        fmt_mean_se(merged_df["unreg_mae_test"]),
        fmt_mean_se(merged_df["reg_best_mae_test"]),
    ],
    "R2_percent": [
        fmt_mean_se(merged_df["stdk_r2_test"], scale=100.0),
        fmt_mean_se(merged_df["unreg_r2_test"], scale=100.0),
        fmt_mean_se(merged_df["reg_best_r2_test"], scale=100.0),
    ],
    "CovFrob": [
        fmt_mean_se(merged_df["stdk_covfrob_test"]),
        fmt_mean_se(merged_df["unreg_covfrob_test"]),
        fmt_mean_se(merged_df["reg_best_covfrob_test"]),
    ],
})

print("\n=== 80/10/10 final table values ===")
print(result_table.to_string(index=False))

print("\n=== Detailed summary (80/10/10, test set) ===")
for model_key, model_name in [
    ("stdk", "STDK"),
    ("unreg", "Adapter (unreg.)"),
    ("reg_best", "Adapter (reg.)"),
]:
    rmse_str = fmt_mean_se(merged_df[f"{model_key}_rmse_test_raw"])
    mae_str = fmt_mean_se(merged_df[f"{model_key}_mae_test"])
    r2_str = fmt_mean_se(merged_df[f"{model_key}_r2_test"], scale=100.0)
    cov_str = fmt_mean_se(merged_df[f"{model_key}_covfrob_test"])

    print(f"{model_name}")
    print(f"  RMSE      = {rmse_str}")
    print(f"  MAE       = {mae_str}")
    print(f"  R^2 (%)   = {r2_str}")
    print(f"  CovFrob   = {cov_str}")

OUT_CSV = REPEAT_DIR / "weather2k_80_10_10_table_values.csv"
result_table.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)

print("\n=== Seed count check ===")
print("n_summary_rows =", len(summary_df))
print("n_npz_files    =", len(npz_files))
print("caption should match this seed count.")

In [ ]:
from spatial_adapter.metrics import rmse_pooled, mae_pooled, r2_pooled, empirical_cov, cov_frob_observed


In [ ]:
summary_df = pd.read_csv(SUMMARY_CSV)
print(summary_df[["seed", "n_sites_full", "n_sites_keep"]].head())
print("mean n_sites_full =", summary_df["n_sites_full"].mean())
print("mean n_sites_keep =", summary_df["n_sites_keep"].mean())